# Neural ODE Post-fit Refinement
This notebook demonstrates running post-fit neural latent-rate refinement and rendering the saved diagnostics inline.


In [ ]:
# Import notebook dependencies and project entrypoints
from __future__ import annotations

import os
from types import SimpleNamespace

import numpy as np
from IPython.display import Image, display

from phoscrosstalk.config import ModelDims
from phoscrosstalk.dashboard import load_fitted_params, load_preopt_snapshot
from phoscrosstalk.derived_rates import make_k_act_fn, make_s_prod_fn
from phoscrosstalk.neuralODE import run_neural_latent_rate_refinement as run_neural_ode
from phoscrosstalk.neuralODE import save_neural_ode_plots
from phoscrosstalk.optimization import NetworkProblem, create_bounds


In [ ]:
# Define run configuration and hyperparameters
RESULTS_DIR = "../thesis_dist"
OUTDIR = os.path.join(RESULTS_DIR, "neural_ode")
NEURAL_CFG = {
    "width": 64, "depth": 3, "steps": 2000, "learning_rate": 1e-3,
    "seed": 42, "print_every": 100, "use_optax": True,
    "prior_weight_k_act": 1.0, "prior_weight_s_prod": 1.0,
    "data_weight_phospho": 1.0, "data_weight_abundance": 0.5, "data_weight_mrna": 0.0,
    "rtol": 1e-5, "atol": 1e-6, "dt0": 0.01, "max_steps": 16384,
    "optimizer": "adabelief", "learn_theta": False,
}
MECHANISM = "dist"
RNA_RELAX = 0.1
ABUNDANCE_MAX = 5.0
neural_cfg = SimpleNamespace(**NEURAL_CFG)
os.makedirs(OUTDIR, exist_ok=True)


In [ ]:
# Load fitted parameters and pre-optimisation snapshot arrays
params = load_fitted_params(RESULTS_DIR)
snap = load_preopt_snapshot(RESULTS_DIR)
if params is None or snap is None:
    raise FileNotFoundError(f"Missing fitted outputs under {RESULTS_DIR!r}.")

theta_best = np.asarray(params["theta"], dtype=np.float64)
proteins = [str(x) for x in params.get("proteins", [])]
sites = [str(x) for x in params.get("sites", [])]
kinases = [f"Kinase_{i}" for i in range(int(np.asarray(snap["R"]).shape[0]))]

t = np.asarray(snap["t"], dtype=np.float64)
P_scaled = np.asarray(snap["P_scaled"], dtype=np.float64)
A_scaled = np.asarray(snap["A_scaled"], dtype=np.float64)
W_data = np.asarray(snap["W_data"], dtype=np.float64)
W_data_prot = np.asarray(snap["W_data_prot"], dtype=np.float64)


In [ ]:
# Build dims, derived-rate closures, and optimization problem inputs
Cg = np.asarray(snap["Cg"], dtype=np.float64)
Cl = np.asarray(snap["Cl"], dtype=np.float64)
site_prot_idx = np.asarray(snap["site_prot_idx"], dtype=np.int32)
K_site_kin = np.asarray(snap["K_site_kin"], dtype=np.float64)
R = np.asarray(snap["R"], dtype=np.float64)
L_alpha = np.asarray(snap["L_alpha"], dtype=np.float64)
kin_to_prot_idx = np.asarray(snap["kin_to_prot_idx"], dtype=np.int32)
receptor_mask_prot = np.asarray(snap["receptor_mask_prot"], dtype=np.float64)
receptor_mask_kin = np.asarray(snap["receptor_mask_kin"], dtype=np.float64)
prot_idx_for_A = np.asarray(snap["prot_idx_for_A"], dtype=np.int32)

dims = ModelDims.from_data(P_scaled, A_scaled if A_scaled.size > 0 else None, kin_to_prot_idx)
xl, xu, _ = create_bounds(dims.K, dims.M, dims.N)
tf_weights = np.asarray(snap["tf_prot_weights"], dtype=np.float64) if "tf_prot_weights" in snap else None
self_rna_idx = np.asarray(snap["protein_self_rna_idx"], dtype=np.int32) if "protein_self_rna_idx" in snap else None
t_rna = np.asarray(snap["t_rna"], dtype=np.float64) if "t_rna" in snap else None


In [ ]:
# Construct rate closures and network problem used by neural refinement
rna_obs = np.asarray(snap["rna_obs_matched"], dtype=np.float64) if "rna_obs_matched" in snap else None
rna_model_prot_idx = np.asarray(snap["rna_model_prot_idx"], dtype=np.int32) if "rna_model_prot_idx" in snap else None
W_data_mrna = np.asarray(snap["W_data_mrna_matched"], dtype=np.float64) if "W_data_mrna_matched" in snap else None
R_data0 = np.asarray(snap["R_data0"], dtype=np.float64) if "R_data0" in snap else None

k_act_fn = make_k_act_fn(t_rna=t_rna, rna_data=rna_obs, tf_prot_weights=tf_weights, K=dims.K, protein_self_rna_idx=self_rna_idx)
s_prod_fn = make_s_prod_fn(t_protein=t, Y_data=P_scaled, R_kin_site=R, kin_to_prot_idx=kin_to_prot_idx, K=dims.K, M=dims.M)

problem = NetworkProblem(
    dims=dims, t=t, P_data=P_scaled, Cg=Cg, Cl=Cl, site_prot_idx=site_prot_idx,
    K_site_kin=K_site_kin, R=R, A_scaled=A_scaled, prot_idx_for_A=prot_idx_for_A,
    W_data=W_data, W_data_prot=W_data_prot, L_alpha=L_alpha, kin_to_prot_idx=kin_to_prot_idx,
    lambda_net=0.0, reg_lambda=0.0, receptor_mask_prot=receptor_mask_prot, receptor_mask_kin=receptor_mask_kin,
    mechanism=MECHANISM, xl=xl, xu=xu, k_act_fn=k_act_fn, s_prod_fn=s_prod_fn, t_rna=t_rna,
    rna_obs_matched=rna_obs, rna_model_prot_idx=rna_model_prot_idx, W_data_mrna=W_data_mrna, R_data0=R_data0,
)


In [ ]:
# Run neural ODE refinement and unpack training outputs
ts, ys, model, loss_history, time_history = run_neural_ode(
    dims=dims,
    problem=problem,
    theta_best=theta_best,
    k_act_fn=k_act_fn,
    s_prod_fn=s_prod_fn,
    t=t, P_scaled=P_scaled, A_scaled=A_scaled, prot_idx_for_A=prot_idx_for_A,
    W_data=W_data, W_data_prot=W_data_prot, proteins=proteins, sites=sites, kinases=kinases,
    t_rna=t_rna, rna_obs_matched=rna_obs, rna_model_prot_idx=rna_model_prot_idx,
    W_data_mrna_matched=W_data_mrna, outdir=OUTDIR, neural_cfg=neural_cfg,
    mechanism=MECHANISM, rna_relax=RNA_RELAX, abundance_max=ABUNDANCE_MAX, R_data0=R_data0,
)


In [ ]:
# Save neural ODE figures and display generated PNG files inline
save_neural_ode_plots(outdir=OUTDIR, ts=ts, ys=ys, model=model, loss_history=loss_history, time_history=time_history)
for name in [
    "neural_ode_training_loss.png",
    "neural_ode_step_time.png",
    "neural_ode_trajectories.png",
]:
    image_path = os.path.join(OUTDIR, name)
    if os.path.exists(image_path):
        display(Image(filename=image_path))


The neural refinement should reduce trajectory mismatch while keeping rates near mechanistic priors. Inspect the three saved plots to check convergence, runtime stability, and trajectory quality.
